# Jev 4b QLoRA — Colab PAYG/Enterprise run (needs L4 or A100)

Calibrated fine-tune of **gemma-3-4b-it** for the confirm-gate
noul decision — produces a real Jev-style model, not the frozen-
feature attention head. Expected cost: ~$1-4 (PAYG compute units)
or ~$0.8-4/hr (Enterprise L4/A100). **Free-tier T4 will likely OOM
or get disconnected** — 4b + LoRA wants ≥16GB and T4 sessions die
unpredictably.

Steps:
1. Runtime → Change runtime type → **A100** (L4 works too, slower).
2. Cell 2 uploads `train-all.jsonl` (in `scripts/jev/` of the chaba
   repo, or `tony-omen:~/jev-retrain-20261002/`).
3. Run all — SFT ~30-60min on A100, then the embedded 32-case
   failure-set eval prints the benchmark vs the gate thr 0.75.
4. Last cell downloads the LoRA adapter (~100-300MB) — merge into
   the base on idc02/omen for serving.

The task is framed as noul: render the same prompt open-jev uses
and train the model to answer `yes`/`no`. Output is then P(yes)
from the softmax over the two option tokens — same contract the
production gate already reads.

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/2**30,1)
      if torch.cuda.is_available() else 0)
!pip -q install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' 2>/dev/null || \
  pip -q install unsloth bitsandbytes accelerate peft trl transformers


In [ ]:
from google.colab import files
import json, io
up = files.upload()                 # pick train-all.jsonl
rows = [json.loads(l) for l in io.BytesIO(next(iter(up.values()))).read()
        .decode().splitlines() if l.strip()]
print(len(rows), 'rows, pos:', sum(1 for r in rows if r['label']))

CONFIRM_STATE = ('Ada, a voice assistant, asked the user to confirm a '
                 'memory write. The user turn was: "{turn}"')
INSTR = ('Did the user explicitly affirm or confirm? The turn counts as '
         'affirmation only when it is a standalone short affirmation '
         '(like yes, ok, confirm, go ahead, ยืนยัน) or begins with an '
         'affirmation. An approval word embedded inside a longer request '
         'does NOT count.')
def render(turn):
    return (f'State:\n{CONFIRM_STATE.format(turn=turn)}\n\n'
            f'Question:\n{INSTR}\n\n'
            f'Answer yes when:\nthe whole turn is a short affirmation, or it leads with one\n\n'
            f'Answer no when:\nno affirmation present, or an affirmative word is buried inside a longer request\n\n'
            'Answer yes or no.\n\nAnswer:\n')

texts = [render(r['text']) + (' yes' if r['label'] else ' no')
         for r in rows]
print(texts[0][-400:])


In [ ]:
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained(
    model_name='unsloth/gemma-3-4b-it',
    max_seq_length=640, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth')


In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
ds = Dataset.from_dict({'text': texts})
trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=ds,
    dataset_text_field='text', max_seq_length=640,
    args=TrainingArguments(
        output_dir='/tmp/qlora', per_device_train_batch_size=8,
        gradient_accumulation_steps=4, num_train_epochs=2,
        learning_rate=2e-4, warmup_steps=20, logging_steps=20,
        save_strategy='no', report_to=[], bf16=True, seed=7))
trainer.train()
model.save_pretrained('/content/jev-4b-lora')
tok.save_pretrained('/content/jev-4b-lora')


In [ ]:
# failure-set eval — P(yes) from softmax over option tokens
import json, time, torch
EVAL = [
 {
  "text": "yes, publish it",
  "label": true
 },
 {
  "text": "Yes",
  "label": true
 },
 {
  "text": "go ahead",
  "label": true
 },
 {
  "text": "ยืนยันครับ",
  "label": true
 },
 {
  "text": "ใช่ ทำเลย",
  "label": true
 },
 {
  "text": "remember that the lab stack design is approved",
  "label": false
 },
 {
  "text": "turn off the TV ok",
  "label": false
 },
 {
  "text": "what is the weather today",
  "label": false
 },
 {
  "text": "save this to memory please",
  "label": false
 },
 {
  "text": "confirm delete the old page",
  "label": false
 },
 {
  "text": "Confirm the camera actually moved — ask it for the current view state.",
  "label": false
 },
 {
  "text": "เออถ้าไหนว่าไป",
  "label": true
 },
 {
  "text": "Yes, publish it.",
  "label": true
 },
 {
  "text": "Yes, publish the Thai version.",
  "label": true
 },
 {
  "text": "Yes, go ahead.",
  "label": true
 },
 {
  "text": "Yes, move the wall if it's busy — do the split.",
  "label": true
 },
 {
  "text": "Yes — do the split and put God's Eye View on the left pane.",
  "label": true
 },
 {
  "text": "Yes — set up the split and load God's Eye View on the left.",
  "label": true
 },
 {
  "text": "อ่ะ เล่น ทับ ไป เลย",
  "label": true
 },
 {
  "text": "รีเซ็ตแป๊บนึง รีเซ็ตไปเลย",
  "label": true
 },
 {
  "text": "อะ ตกลง ครับ",
  "label": true
 },
 {
  "text": "ตกลง ครับ",
  "label": true
 },
 {
  "text": "ปิด ไป เลย ปิด ไป เลย กล้อง วงจรปิด",
  "label": true
 },
 {
  "text": "ปิดไปเลยแล้วส่ง อ่า GEV มาแทน Thonburi TV",
  "label": true
 },
 {
  "text": "ให้ ดำเนิน การ ได้ เลย",
  "label": true
 },
 {
  "text": "ค่ะ เผยแพร่เลย ขอบคุณค่ะ",
  "label": true
 },
 {
  "text": "ตกลง",
  "label": true
 },
 {
  "text": "อ่าเริ่มได้เลยครับผม",
  "label": true
 },
 {
  "text": "ที่บ้านผมนี่แหละ เออ ของคุณเคเคแต่อยู่ที่บ้านผมนะ อัพเดทข้อมูลให้ที",
  "label": false
 },
 {
  "text": "เปิดวอลเปเปอร์",
  "label": false
 },
 {
  "text": "ส่ง Camwall ตัวต่อไปมาที่ทีวี",
  "label": false
 },
 {
  "text": "อามิ ส่ง Camp Wall ตัวต่อไปมาที่ทีวี",
  "label": false
 }
]
FastLanguageModel.for_inference(model)
yes_id = tok(' yes', add_special_tokens=False)['input_ids'][-1]
no_id  = tok(' no',  add_special_tokens=False)['input_ids'][-1]
res=[]
for r in EVAL:
    ids = tok(render(r['text']), return_tensors='pt').to('cuda')
    t=time.time()
    with torch.no_grad():
        logits = model(**ids).logits[0,-1]
    p = float(torch.softmax(torch.stack([logits[yes_id], logits[no_id]]),0)[0])
    res.append({**r,'p':p,'s':time.time()-t,'ok':(p>=0.75)==r['label']})
import numpy as np
acc=lambda sel:round(np.mean([x['ok'] for x in res if sel(x)]),3)
thai=lambda x:any('\u0e00'<=c<='\u0e7f' for c in x['text'])
print('acc',acc(lambda x:True),'thai',acc(thai),'pos',acc(lambda x:x['label']),
      'neg',acc(lambda x:not x['label']),'lat_ms',round(np.median([x['s'] for x in res])*1000))
for x in res:
    if not x['ok']: print(' MISS',round(x['p'],3),x['label'],x['text'][:60])


In [ ]:
!cd /content && zip -qr jev-4b-lora.zip jev-4b-lora
files.download('/content/jev-4b-lora.zip')
